<a href="https://colab.research.google.com/github/radwasafty-spec/IEEE-CS-DataScience-26/blob/main/T15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Import necessary libraries

In [1]:
import pandas as pd
import numpy as np

### Step 2: Load the Data

In [8]:
from google.colab import files
uploaded = files.upload()

Saving cafe_sales.csv to cafe_sales (1).csv


### Step 3: Explore the Data

##### display the first 10 rows to see the "messy" data

In [10]:
df = pd.read_csv('cafe_sales (1).csv')
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


##### check how many rows and columns

In [11]:
df.shape

(10000, 8)

##### display basic info about the dataset to identify nulls and incorrect data types

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


##### check unique values for `Item`, `Payment Method`, and `Location`

In [13]:
print(df['Item'].unique())
print(df['Payment Method'].unique())
print(df['Location'].unique())

['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]
['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']


##### check missing values

In [14]:
df.isnull().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


### Step 4: Handle Missing Values

##### Spot strings like "UNKNOWN" and "ERROR" that Pandas doesn't see as nulls yet

In [15]:
print(df['Item'].value_counts())
print(df['Payment Method'].value_counts())
print(df['Location'].value_counts())

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
ERROR        292
Name: count, dtype: int64
Payment Method
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64
Location
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


##### Use `.replace()` to convert "UNKNOWN" and "ERROR" to np.nan

In [16]:
df.replace(['UNKNOWN', 'ERROR'], np.nan, inplace=True)

### Step 5:Text Sanitization

##### remove hidden spaces from all string columns

In [17]:
string_cols = df.select_dtypes(include=['object']).columns

for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

##### convert all Item names to lowercase for consistency

In [19]:
df["Item"] = df["Item"].str.lower()

##### use a dictionary to fix any typos in item names if they exist

In [24]:
typo_dict = {
    'appl': 'apple',
    'bnn': 'banana',
    'electrnics': 'electronics'

}

df['Item'] = df['Item'].replace(typo_dict)

### Step 6: Fix Data Types
##### fix incorrect data types (Quantity, Price Per Unit, Total Spent .....)

In [25]:
import pandas as pd


df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')

##### identify and fix rows where Quantity might be zero or negative

In [26]:
df = df[df['Quantity'] > 0]

### Step 7: Handle Missing Values (Part 2: Fill NaNs)

##### fill missing `Price Per Unit`

In [28]:
df.loc[:, 'Price Per Unit'] = df['Price Per Unit'].fillna(df.groupby('Item')['Price Per Unit'].transform('mean'))
df.loc[:, 'Price Per Unit'] = df['Price Per Unit'].fillna(df['Price Per Unit'].mean())

##### handle other missing values appropriately (like Payment Method or Location)

In [30]:
df.loc[:, 'Payment Method'] = df['Payment Method'].fillna('Unknown')
df.loc[:, 'Location'] = df['Location'].fillna('Unknown')

### Step 8: Logical Validation

##### create a new column `Expected_Total` by multiplying `Quantity * Price Per Unit`

In [34]:
df.loc[:, 'Expected_Total'] = df['Quantity'] * df['Price Per Unit']

##### filter rows where Total Spent is NOT equal to Expected_Total

In [35]:
df[df['Total Spent'] != df['Expected_Total']]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Expected_Total
2,TXN_4271903,cookie,4.0,1.000000,NaN,Credit Card,In-store,2023-07-19,4.00000
25,TXN_7958992,smoothie,3.0,4.000000,NaN,nan,nan,2023-12-13,12.00000
31,TXN_8927252,nan,2.0,1.000000,NaN,Credit Card,nan,2023-11-06,2.00000
42,TXN_6650263,tea,2.0,1.500000,NaN,nan,Takeaway,2023-01-10,3.00000
65,TXN_4987129,sandwich,3.0,4.000000,NaN,nan,In-store,2023-10-20,12.00000
...,...,...,...,...,...,...,...,...,...
9954,TXN_1191659,coffee,4.0,2.000000,NaN,Credit Card,In-store,2023-11-21,8.00000
9977,TXN_5548914,juice,2.0,3.000000,NaN,Digital Wallet,In-store,2023-11-04,6.00000
9988,TXN_9594133,cake,5.0,3.000000,NaN,nan,nan,nan,15.00000
9993,TXN_4766549,smoothie,2.0,4.000000,NaN,Cash,nan,2023-10-20,8.00000


##### fill missing or incorrect Total Spent values using the Expected_Total calculation.

In [37]:
df.loc[:, 'Total Spent'] = df['Expected_Total']

### Step 9: Handle Dates

##### convert Transaction Date to datetime

In [39]:
df.loc[:, 'Transaction Date'] = pd.to_datetime(df['Transaction Date'])

##### identify transactions with dates in the future (if any) and handle them

In [40]:
df = df[df['Transaction Date'] <= pd.Timestamp.now()]

##### extract "Year" and "Month" into separate columns for analysis

In [42]:
df.loc[:, 'Year'] = df['Transaction Date'].dt.year
df.loc[:, 'Month'] = df['Transaction Date'].dt.month

### Step 10: Remove Duplicates

##### check for duplicates

In [43]:
df.duplicated().sum()

np.int64(0)

##### drop duplicates if found

In [44]:
df = df.drop_duplicates().copy()

### Step 9: Clean Text Data

#####  clean text columns (strip spaces, unify format)

In [45]:
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    df.loc[:, col] = df[col].astype(str).str.strip().str.lower()